# Paper 2: real Allow and Automated prompts for the v2 website pool (WildChat-4.8M, non-toxic)

**What this does.** The v2 pool (`deploy/prompt_bank_v2.json`) is 1,000 prompts: 700 invented Shadow AI prompts (`make_pool_v2_shadow.py`), 200 real Allow and 100 real Automated prompts from this notebook. Pool prompts are shown to participants and agents on the website, so they must stay out of every training set and must not carry anyone's personal data.

**How the 300 real prompts are drawn.** One streaming pass over `allenai/WildChat-4.8M` (the non-toxic release) keeps a **uniform random** 1-in-`KEEP_DEN` sample of conversations, chosen by a hash of the conversation id, so the draw is spread over the whole archive and is reproducible. There is no pattern search. A sampled first-turn English prompt is kept only if:
- the conversation is not toxic, not redacted, and its first turn was not flagged by moderation;
- it is 20 to 900 characters (a person can retype it); prompts of 900 to 2,500 characters are also drawn, at a higher rate (1 in `LONG_KEEP_DEN`, a superset of the base sample), for the Automated quota (most Automated prompts are long templates) and to top up work-style Allow (mostly pasted work text); general Allow stays under 900 characters;
- `mask_secrets.py` finds **nothing** in it (no emails, phones, keys, card numbers and so on), so no personal data reaches the site;
- its id is not in the 10K labeled sample, the audit key, the pass-1 candidates, the verified set, the human gold, the base training corpus or the v1 pool, and it is not a near-duplicate (MinHash 0.85) of any training, audit or v1-pool prompt;
- at most one prompt per hashed IP.

The kept prompts are labeled by the same judge (claude-fable-5) under rubric v3.1 (hash `c382d28af8`, checked). The invented Shadow AI part is labeled too, as a check that the judge reads it as Shadow AI.

**Then you review** `audit/pool_v2_review.xlsx` (cell 7) and run the assembly (cell 8), which writes `deploy/prompt_bank_v2.json`.

**After a runtime restart** run cells 1 and 2 first (they define names the later cells use; cell 3 is needed only while streaming); every later cell reloads its inputs from Drive, and cell 4 recognizes a finished stream and skips it.

**Source:** this notebook and the scripts it calls are kept in the faylynx repository under `study/pool_v2/`; copy the scripts into `MyDrive/paper2-three-class/code/` before running.

**Order:** run cells 1 to 7 top to bottom, review the workbook, then run cell 8. Cell 1b joins the base corpus from its two uploaded parts the first time (later runs skip it). Cell 1c clears an unfinished earlier run only when its box is ticked; leave it unticked to resume after a disconnect. The code and data files this needs are already on Drive (`code/mask_secrets.py`, `code/build_prompt_bank_v2.py`, `deploy/prompt_bank_v2_shadow_part.json`, `deploy/prompt_bank.json`, `rubric/judge_prompt_v3_1.txt`, and the base corpus parts).

**Secrets:** `ANTHROPIC_API_KEY`, `HF_TOKEN` in Colab Secrets. **Outputs:** `data/raw/pool_v2_real_candidates.jsonl`, `data/labeled/pool_v2_real_fable_v3_1.jsonl`, `data/labeled/pool_v2_shadow_part_fable_v3_1.jsonl`, `audit/pool_v2_review.xlsx` (+ `_key.json`), and after review `deploy/prompt_bank_v2.json`.

A full pass takes about five to six hours (about 14,000 conversations a minute); cell 4 checkpoints every 200,000 conversations and resumes from there.


In [ ]:
#@title 1) Install, mount Drive, secrets, config
!pip -q install anthropic datasets huggingface_hub pyarrow openpyxl datasketch
import os, re, sys, gzip, json, time, random, hashlib
from collections import Counter, defaultdict
from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

ROOT        = "/content/drive/MyDrive/paper2-three-class"   #@param {type:"string"}
DATASET     = "allenai/WildChat-4.8M"   #@param ["allenai/WildChat-4.8M"]
KEEP_DEN    = 1000        #@param {type:"integer"}   # keep about 1 in KEEP_DEN conversations (by hash of the conversation id)
LONG_KEEP_DEN = 250       #@param {type:"integer"}   # 900-2,500 character prompts are kept at 1 in LONG_KEEP_DEN (must divide KEEP_DEN); used for Automated only
MAX_CANDIDATES = 3000     #@param {type:"integer"}   # safety cap; a full pass is expected to keep about 2,100
MAX_STREAM  = 0           #@param {type:"integer"}   # 0 = whole archive
MIN_CHARS, MAX_CHARS, AUTO_MAX_CHARS = 20, 900, 2500   # Allow must be <= MAX_CHARS (retypeable); Automated may be up to AUTO_MAX_CHARS
assert KEEP_DEN % LONG_KEEP_DEN == 0, "LONG_KEEP_DEN must divide KEEP_DEN so the long sample contains the base sample"
SEED        = 20261120    #@param {type:"integer"}
QUOTA = {"Allow_work": 150, "Allow_general": 50, "Automated AI Pipelines": 100}
RESERVE_FRAC = 0.5        # extra candidates per group shown in the review workbook as replacements
MIN_JUDGE_CONF = 0.8      # only confident judge labels go into the pool

sys.path.insert(0, f"{ROOT}/code")
from mask_secrets import mask, MASK_VERSION
RAW_DIR, LAB_DIR, AUD_DIR = f"{ROOT}/data/raw", f"{ROOT}/data/labeled", f"{ROOT}/audit"
for d in (RAW_DIR, LAB_DIR, AUD_DIR): os.makedirs(d, exist_ok=True)
CAND_PATH  = f"{RAW_DIR}/pool_v2_real_candidates.jsonl"
CKPT_PATH  = CAND_PATH + ".checkpoint.json"
OUT_PATH   = f"{LAB_DIR}/pool_v2_real_fable_v3_1.jsonl"
SHADOW_PART = f"{ROOT}/deploy/prompt_bank_v2_shadow_part.json"
SHADOW_OUT  = f"{LAB_DIR}/pool_v2_shadow_part_fable_v3_1.jsonl"
REVIEW_XLSX = f"{AUD_DIR}/pool_v2_review.xlsx"
for p in (f"{ROOT}/code/mask_secrets.py", SHADOW_PART, f"{ROOT}/code/build_prompt_bank_v2b.py", f"{ROOT}/code/pool_v2_defang.py"):
    assert os.path.exists(p) or p.endswith("build_prompt_bank_v2.py"), f"copy {p.replace(ROOT + '/', '')} from the laptop to Drive first"

ANTHROPIC_KEY = userdata.get("ANTHROPIC_API_KEY"); assert ANTHROPIC_KEY, "add ANTHROPIC_API_KEY to Colab Secrets"
try: HF_TOKEN = userdata.get("HF_TOKEN")
except Exception: HF_TOKEN = None
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)
print("mask:", MASK_VERSION, "| candidates ->", CAND_PATH, "| labels ->", OUT_PATH)


In [ ]:
#@title 1b) One-time setup: join the base corpus from its two parts (does nothing once deploy/base_corpus.jsonl.gz exists)
import shutil
BC = f"{ROOT}/deploy/base_corpus.jsonl.gz"
PARTS = [f"{ROOT}/deploy/base_corpus.part1.jsonl.gz", f"{ROOT}/deploy/base_corpus.part2.jsonl.gz"]
if os.path.exists(BC):
    print("base corpus already in place:", os.path.getsize(BC), "bytes")
elif all(os.path.exists(p) for p in PARTS):
    # same as: !cd "/content/drive/MyDrive/paper2-three-class/deploy" && cat base_corpus.part1.jsonl.gz base_corpus.part2.jsonl.gz > base_corpus.jsonl.gz
    with open(BC, "wb") as out:
        for p in PARTS:
            with open(p, "rb") as f: shutil.copyfileobj(f, out)
    want = sum(os.path.getsize(p) for p in PARTS)
    assert os.path.getsize(BC) == want, "joined file has the wrong size; delete it and rerun this cell"
    n = sum(1 for l in gzip.open(BC, "rt", encoding="utf-8") if l.strip())
    print(f"joined base corpus: {os.path.getsize(BC)} bytes, {n:,} lines (expect 3,457,096 bytes and 11,465 lines); the two part files can now be deleted")
else:
    raise FileNotFoundError("deploy/base_corpus.jsonl.gz is missing and so are its two parts; copy the file from the laptop's deploy folder")


In [ ]:
#@title 1c) Start fresh (tick only to discard an earlier, unfinished cell-4 run; leave unticked when resuming after a disconnect)
CLEAR_EARLIER_RUN = False   #@param {type:"boolean"}
if CLEAR_EARLIER_RUN:
    get_ipython().system('cd "/content/drive/MyDrive/paper2-three-class/data/raw" && rm -f pool_v2_real_candidates.jsonl pool_v2_real_candidates.jsonl.checkpoint.json && echo cleared')
    print("Untick CLEAR_EARLIER_RUN now, so rerunning all cells after a disconnect resumes instead of starting over.")
else:
    have = os.path.exists(CAND_PATH)
    print("keeping any earlier run:", f"{sum(1 for _ in open(CAND_PATH, encoding='utf-8')):,} candidates on file, cell 4 will resume" if have else "none on file, cell 4 starts from the beginning")


In [ ]:
#@title 2) Rubric v3.1 from rubric/judge_prompt_v3_1.txt (hash checked), reply parser and batch helper (verbatim from paper2_mine_shadow_v3_1.ipynb cell 3)
CLASSES = ["Allow", "Shadow AI", "Automated AI Pipelines"]
VALID = set(CLASSES) | {"Unsure"}
SYSTEM = open(f"{ROOT}/rubric/judge_prompt_v3_1.txt", encoding="utf-8").read()
PROMPT_VERSION = "judge_prompt_v3_1:" + hashlib.sha256(SYSTEM.encode()).hexdigest()[:10]
assert PROMPT_VERSION == "judge_prompt_v3_1:c382d28af8", f"rubric file changed ({PROMPT_VERSION}); labels would not be comparable with pass 1"
print(f"system prompt ~{len(SYSTEM)//4} tokens | {PROMPT_VERSION}")

def parse_label_reply(text, valid, want_turn=False):
    t = (text or "").strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S)
    m = re.search(r"\{.*\}", t, re.S)
    if not m: return None
    try: d = json.loads(m.group(0))
    except Exception: return None
    lab = d.get("label")
    if lab not in valid: return None
    try: conf = max(0.0, min(1.0, float(d.get("confidence", 0.5))))
    except Exception: conf = 0.5
    rec = {"label": lab, "confidence": conf}
    if want_turn:
        try: rec["trigger_turn"] = int(d.get("trigger_turn", 1))
        except Exception: rec["trigger_turn"] = None
    return rec


import anthropic
FRONTIER_MODEL = "claude-fable-5"      #@param {type:"string"}
AN_PRICE_IN, AN_PRICE_OUT, AN_PRICE_CR, AN_PRICE_CW = 5.0, 25.0, 0.5, 6.25   # batch rates per million tokens; verify at https://claude.com/pricing
an_client = anthropic.Anthropic(api_key=ANTHROPIC_KEY) if ANTHROPIC_KEY else None

def label_with_anthropic_batch(rows, out_path, system, model=None, valid=None, max_chars=4000, wave_size=4000, poll_secs=30, want_turn=False, text_key="prompt"):
    """Same contract as label_with_openai_batch. Needs ANTHROPIC_API_KEY in Colab Secrets."""
    assert an_client is not None, "ANTHROPIC_API_KEY is not set in Colab Secrets"
    model = model or FRONTIER_MODEL; valid = valid or set(CLASSES) | {"Unsure"}
    pending = out_path + ".pending.json"
    def labeled():
        done = {}
        if os.path.exists(out_path):
            for l in open(out_path, encoding="utf-8"):
                try: r = json.loads(l)
                except Exception: continue
                if r.get("label") in valid or r.get("label") == "REFUSED": done[r["prompt_id"]] = r   # a refusal is the judge's final answer; do not resubmit
        return done
    def make_request(cid, text):
        return {"custom_id": cid, "params": {"model": model, "max_tokens": 600, "output_config": {"effort": "low"},
                "system": [{"type": "text", "text": system, "cache_control": {"type": "ephemeral"}}],
                "messages": [{"role": "user", "content": (text or "")[:max_chars]}]}}
    def poll(bid):
        delay, last = poll_secs, None
        while True:
            b = an_client.messages.batches.retrieve(bid); rc = b.request_counts
            state = (b.processing_status, rc.succeeded, rc.errored, rc.processing)
            if state != last: print(f"  [{bid[-8:]}] {b.processing_status} | ok {rc.succeeded} | err {rc.errored} | processing {rc.processing}", flush=True); last = state
            if b.processing_status == "ended": return b
            time.sleep(delay); delay = min(delay * 1.5, 300)
    def collect(bid, row_map):
        tk = Counter(); n = 0
        with open(out_path, "a", encoding="utf-8") as out:
            for e in an_client.messages.batches.results(bid):
                row = row_map.get(e.custom_id)
                if row is None: continue
                if e.result.type == "succeeded":
                    msg = e.result.message; u = msg.usage
                    tk["in"] += u.input_tokens; tk["out"] += u.output_tokens
                    tk["cr"] += getattr(u, "cache_read_input_tokens", 0) or 0; tk["cw"] += getattr(u, "cache_creation_input_tokens", 0) or 0
                    text = "".join(blk.text for blk in msg.content if getattr(blk, "type", "") == "text")
                    stop = getattr(msg, "stop_reason", "?")
                    parsed = parse_label_reply(text, valid, want_turn) or {"label": "REFUSED" if stop == "refusal" else "ERROR", "confidence": 0.0, "error": f"unparseable (stop_reason={stop})"}
                    rec = {**row, "judge_model": msg.model, **parsed, "in_tok": u.input_tokens, "out_tok": u.output_tokens}
                else:
                    rec = {**row, "judge_model": model, "label": "ERROR", "confidence": 0.0, "error": f"{e.result.type}: {str(getattr(e.result, 'error', ''))[:120]}"}
                out.write(json.dumps(rec, ensure_ascii=False) + "\n"); n += 1
        usd = tk["in"] / 1e6 * AN_PRICE_IN + tk["out"] / 1e6 * AN_PRICE_OUT + tk["cr"] / 1e6 * AN_PRICE_CR + tk["cw"] / 1e6 * AN_PRICE_CW
        print(f"  collected {n} | tokens in/out {tk['in']:,}/{tk['out']:,} | cache read/write {tk['cr']:,}/{tk['cw']:,} | est ${usd:.2f}")
    if os.path.exists(pending):
        p = json.load(open(pending)); print("resuming pending batch", p["batch_id"]); poll(p["batch_id"]); collect(p["batch_id"], p["rows"]); os.remove(pending)
    while True:
        done = labeled(); todo = [r for r in rows if r["prompt_id"] not in done]
        print(f"labeled {len(done)} / {len(rows)} | remaining {len(todo)}")
        if not todo: break
        wave = todo[:wave_size]; row_map = {f"x{i}": r for i, r in enumerate(wave)}
        b = an_client.messages.batches.create(requests=[make_request(cid, r[text_key]) for cid, r in row_map.items()])
        json.dump({"batch_id": b.id, "rows": row_map}, open(pending, "w")); print(f"  submitted {b.id} with {len(wave)} requests")
        poll(b.id); collect(b.id, row_map); os.remove(pending)
        if len(labeled()) == len(done): print("  no new labels from this wave; stopping"); break
    return labeled()
print("Anthropic batch helper ready:", FRONTIER_MODEL if an_client else "(no key; frontier labeling disabled)")


In [ ]:
#@title 3) What the pool must stay away from: ids and near-duplicates of training, audit, pass-1, verified, gold and v1-pool prompts
from datasketch import MinHash, MinHashLSH
def norm(t): return re.sub(r"\s+", " ", (t or "").strip().lower())
def read_jsonl(p, opener=open):
    with opener(p, "rt", encoding="utf-8") as f:
        for l in f:
            if l.strip():
                try: yield json.loads(l)
                except Exception: pass
def rid(r): return str(r.get("prompt_id") or r.get("id") or "")
def conv_id(h): return f"wc48m:{h}"

excl_ids, excl_texts, src_counts = set(), [], Counter()
def add(path, opener=open, text=True, label=None):
    if not os.path.exists(path): print("  (not found, skipped)", path.replace(ROOT + "/", "")); return
    n = 0
    for r in read_jsonl(path, opener):
        i = rid(r)
        if i: excl_ids.add(i)
        if r.get("conversation_hash"): excl_ids.add(conv_id(r["conversation_hash"]))
        # sealed shadow_test rows: only the id is used (to keep them off the site); their text is not read
        if text and r.get("prompt") and r.get("verify_split") != "shadow_test" and r.get("gold_split") != "shadow_test":
            excl_texts.append(r["prompt"])
        n += 1
    src_counts[label or os.path.basename(path)] = n

add(f"{ROOT}/data/labeled/wildchat48m_full_10k_fable_v3.jsonl")
add(f"{ROOT}/deploy/base_corpus.jsonl.gz", gzip.open)
add(f"{ROOT}/data/gold/shadow_verified_v1.jsonl")
add(f"{ROOT}/data/gold/gold_consolidated_v1_text.jsonl")
add(f"{ROOT}/data/raw/shadow_candidates_v1.jsonl", text=False)       # pass-1 candidates: ids only
add(f"{ROOT}/data/raw/paper2b/wildchat_sample.jsonl")
key_path = f"{ROOT}/data/gold/sealed/human_labeling_1k_key.json"
if os.path.exists(key_path):
    for e in json.load(open(key_path))["key"]: excl_ids.add(str(e["prompt_id"]))     # ids only; no labels are read
bank_v1 = json.load(open(f"{ROOT}/deploy/prompt_bank.json"))
for b in bank_v1["prompts"]:
    excl_ids.add(str(b.get("source_id", "")).replace("gold:", ""))
    excl_texts.append(b["type_text"] + ("\n\n" + b["paste_text"] if b.get("paste_text") else ""))
print("sources:", dict(src_counts))
print(f"excluded ids {len(excl_ids):,} | texts for near-duplicate checks {len(excl_texts):,}")

def shingles(t, k=3):
    w = norm(t).split(); return {" ".join(w[i:i+k]) for i in range(max(1, len(w)-k+1))} or {norm(t)[:32]}
def mh(t):
    m = MinHash(num_perm=64, seed=7); m.update_batch([s.encode() for s in shingles(t)]); return m
lsh = MinHashLSH(threshold=0.85, num_perm=64); t0 = time.time()
for i, t in enumerate(excl_texts): lsh.insert(f"x{i}", mh(t))
excl_norm = {norm(t) for t in excl_texts}
print(f"near-duplicate index built in {time.time()-t0:.0f}s")


In [ ]:
#@title 4) Stream the non-toxic archive, keep a uniform random sample that passes every filter (resumable; one pass)
from datasets import load_dataset
def bucket(h): return int(hashlib.sha1(str(h).encode()).hexdigest()[:8], 16)
def first_user_turn(conv):
    for t in conv or []:
        if (t.get("role") or "").lower() == "user": return t
    return None

def stream(cands, ck):
    seen_ip = Counter(c.get("hashed_ip") for c in cands); seen_text = {norm(c["prompt"]) for c in cands}
    print(f"resuming with {len(cands):,} candidates, skipping the first {ck['skip']:,} conversations")
    ds = load_dataset(DATASET, split="train", streaming=True, token=HF_TOKEN)
    t0 = time.time(); n = 0; why = Counter()
    with open(CAND_PATH, "a", encoding="utf-8") as out:
        for ex in ds:
            n += 1
            if n <= ck["skip"]: continue
            if MAX_STREAM and n > MAX_STREAM: break
            if len(cands) >= MAX_CANDIDATES: break
            if n % 200000 == 0:
                json.dump({"skip": n}, open(CKPT_PATH, "w")); print(f"  {n:,} streamed | {len(cands):,} kept | {(time.time()-t0)/60:.1f} min | {dict(why)}", flush=True)
            h = ex.get("conversation_hash"); b = bucket(h)
            if b % LONG_KEEP_DEN: continue                      # not in even the larger (long-prompt) sample
            base_sampled = (b % KEEP_DEN == 0)
            why["sampled"] += 1
            if ex.get("toxic"): why["toxic"] += 1; continue
            if (ex.get("language") or "") != "English": why["not_english"] += 1; continue
            if conv_id(h) in excl_ids: why["excluded_id"] += 1; continue
            turn = first_user_turn(ex.get("conversation"))
            if turn is None: continue
            text = (turn.get("content") or "").strip()
            if not (MIN_CHARS <= len(text) <= AUTO_MAX_CHARS): why["length"] += 1; continue
            if len(text) <= MAX_CHARS and not base_sampled: why["short_not_in_base_sample"] += 1; continue   # short prompts keep the 1-in-KEEP_DEN rate
            if turn.get("redacted"): why["redacted"] += 1; continue
            mod = (ex.get("openai_moderation") or [None])[0] or {}
            if isinstance(mod, dict) and mod.get("flagged"): why["moderation_flagged"] += 1; continue
            if mask(text)[1]: why["has_personal_data_or_secret"] += 1; continue      # nothing maskable may reach the site
            k_ = norm(text)
            if k_ in excl_norm or k_ in seen_text or lsh.query(mh(text)): why["near_duplicate"] += 1; continue
            hip = turn.get("hashed_ip") or ""
            if hip and seen_ip[hip] >= 1: why["ip_cap"] += 1; continue
            row = {"id": conv_id(h), "prompt_id": conv_id(h), "prompt": text, "conversation_hash": h, "model": ex.get("model"),
                   "month": str(ex.get("timestamp"))[:7], "hashed_ip": hip, "source": "wildchat48m_pool_v2", "stream_position": n,
                   "length_band": "short" if len(text) <= MAX_CHARS else "long"}
            out.write(json.dumps(row, ensure_ascii=False) + "\n"); out.flush()
            cands.append(row); seen_text.add(k_); seen_ip[hip] += 1; why["kept"] += 1
    json.dump({"skip": n, "finished": True}, open(CKPT_PATH, "w"))
    print(f"done: {len(cands):,} candidates after {n:,} conversations | {(time.time()-t0)/60:.1f} min | {dict(why)}")

cands = list(read_jsonl(CAND_PATH)) if os.path.exists(CAND_PATH) else []
ck = json.load(open(CKPT_PATH)) if os.path.exists(CKPT_PATH) else {"skip": 0}
# mid-run checkpoints land on exact multiples of 200,000; the end-of-run checkpoint does not (and newer runs also mark it finished)
STREAM_DONE = bool(ck.get("finished")) or (ck["skip"] > 0 and ck["skip"] % 200000 != 0) or len(cands) >= MAX_CANDIDATES
if STREAM_DONE:
    print(f"streaming already finished: {len(cands):,} candidates from {ck['skip']:,} conversations; go on to cell 5")
    print("by length:", dict(Counter(c.get("length_band", "short") for c in cands)))
else:
    stream(cands, ck)


In [ ]:
#@title 5) Label the real candidates with claude-fable-5 under rubric v3.1 (Batch API, resume-safe)
cands = [json.loads(l) for l in open(CAND_PATH, encoding="utf-8") if l.strip()]   # from the file, so a runtime restart after cell 4 is harmless
print(f"labeling {len(cands):,} real candidates with {FRONTIER_MODEL} | {PROMPT_VERSION}")
label_with_anthropic_batch(cands, OUT_PATH, SYSTEM, valid=VALID)
real = [json.loads(l) for l in open(OUT_PATH, encoding="utf-8") if l.strip()]
for r in real: r.setdefault("prompt_version", PROMPT_VERSION)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for r in real: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("label mix:", dict(Counter(r["label"] for r in real)))
print("confident (>= %.2f):" % MIN_JUDGE_CONF, dict(Counter(r["label"] for r in real if (r.get("confidence") or 0) >= MIN_JUDGE_CONF)))


In [ ]:
#@title 6) Check the invented Shadow AI part with the same judge (does it read as Shadow AI?)
part = json.load(open(SHADOW_PART))["prompts"]
srows = [{"prompt_id": p["id"], "id": p["id"], "prompt": p["type_text"] + ("\n\n" + p["paste_text"] if p["paste_text"] else ""), "kind": p["kind"], "role": p["role"]} for p in part]
label_with_anthropic_batch(srows, SHADOW_OUT, SYSTEM, valid=VALID)
sl = {r["prompt_id"]: r for r in (json.loads(l) for l in open(SHADOW_OUT, encoding="utf-8") if l.strip())}
by_kind = defaultdict(Counter)
for p in part: by_kind[p["kind"]][sl.get(p["id"], {}).get("label", "missing")] += 1
print(f"judge labels on the {len(part)} invented prompts:", dict(Counter(r["label"] for r in sl.values())))
for kind, c in sorted(by_kind.items()):
    tot = sum(c.values()); print(f"  {kind:<20s} Shadow {c['Shadow AI']}/{tot} = {100*c['Shadow AI']/tot:.0f}% | {dict(c)}")


In [ ]:
#@title 7) Review workbook: proposed real prompts (with reserves) and the invented prompts the judge did not call Shadow AI
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
WORK = re.compile(r"\b(?:email|report|manager|client|customer|meeting|presentation|proposal|invoice|policy|project|deadline|budget|"
                  r"spreadsheet|excel|sql|python|code|function|script|data|analysis|team|colleague|boss|interview|resume|cover letter|"
                  r"business|marketing|sales|product|strategy|memo|slides?|agenda|stakeholder|kpi|onboarding|training)\b", re.I)
rng = random.Random(SEED)
real = [json.loads(l) for l in open(OUT_PATH, encoding="utf-8") if l.strip()]          # from files, so a restart after cells 5-6 is harmless
part = json.load(open(SHADOW_PART))["prompts"]
sl = {r["prompt_id"]: r for r in (json.loads(l) for l in open(SHADOW_OUT, encoding="utf-8") if l.strip())}
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
conf = [r for r in real if (r.get("confidence") or 0) >= MIN_JUDGE_CONF and r.get("label") in ("Allow", "Automated AI Pipelines")]
bad = [r for r in conf if ILLEGAL_CHARACTERS_RE.search(r["prompt"])]
conf = [r for r in conf if not ILLEGAL_CHARACTERS_RE.search(r["prompt"])]   # invisible control characters: Excel can't hold them and the site shouldn't show them
print(f"left out {len(bad)} prompts containing control characters")
short = lambda r: len(r["prompt"]) <= MAX_CHARS     # general Allow must be short (retypeable); work-style Allow may be long (mostly pasted work text)
groups = {"Allow_work": [r for r in conf if r["label"] == "Allow" and WORK.search(r["prompt"])],
          "Allow_general": [r for r in conf if r["label"] == "Allow" and short(r) and not WORK.search(r["prompt"])],
          "Automated AI Pipelines": [r for r in conf if r["label"] == "Automated AI Pipelines"]}
# if a group is short, its shortfall moves to the other Allow group so Allow still totals 200
eff = dict(QUOTA)
if len(groups["Allow_work"]) < eff["Allow_work"]:
    gap = eff["Allow_work"] - len(groups["Allow_work"]); eff["Allow_work"] -= gap; eff["Allow_general"] += gap
    print(f"only {len(groups['Allow_work'])} confident work-style Allow: moving {gap} places to general Allow -> quotas {eff}")
elif len(groups["Allow_general"]) < eff["Allow_general"]:
    gap = eff["Allow_general"] - len(groups["Allow_general"]); eff["Allow_general"] -= gap; eff["Allow_work"] += gap
    print(f"only {len(groups['Allow_general'])} confident general Allow: moving {gap} places to work-style Allow -> quotas {eff}")
rows_out, key = [], []
for g, lst in groups.items():
    rng.shuffle(lst)
    if g == "Allow_work": lst.sort(key=lambda r: not short(r))      # short ones first (random order kept within each length band); long ones fill the rest
    want = eff[g]; take = lst[: int(want * (1 + RESERVE_FRAC))]
    if len(lst) < want: print(f"WARNING: only {len(lst)} confident {g} candidates for a quota of {want}; raise MAX_CANDIDATES and rerun cells 4-7")
    for j, r in enumerate(take): rows_out.append((g, r, "Y" if j < want else ""))
wb = Workbook(); ws = wb.active; ws.title = "Real prompts"
ws["A1"] = ("Mark use = Y for prompts that go in the pool, N to drop. The first quota in each group is pre-marked Y; blanks are reserves. "
            "Drop anything with personal details, anything offensive, and anything you would not want a participant to retype. "
            f"Quotas: {eff}. Allow must total {eff['Allow_work'] + eff['Allow_general']}: if you drop one Allow row, mark one Allow reserve Y (either group)."); ws["A1"].font = Font(bold=True)
hdr = ["ref", "group", "judge label", "judge conf", "prompt", "use (Y/N)", "notes", "chars"]
for c, h in enumerate(hdr, 1): ws.cell(row=3, column=c, value=h).font = Font(bold=True); ws.cell(row=3, column=c).fill = PatternFill("solid", fgColor="D9E1F2")
for i, (g, r, use) in enumerate(rows_out, 1):
    ref = f"R{i:04d}"; key.append({"ref": ref, "group": g, "prompt_id": r["prompt_id"], "label": r["label"], "confidence": r.get("confidence"), "prompt_version": r.get("prompt_version")})
    for c, v in enumerate([ref, g, r["label"], r.get("confidence"), r["prompt"], use, "", len(r["prompt"])], 1): ws.cell(row=3+i, column=c, value=v)
    ws.cell(row=3+i, column=5).alignment = Alignment(wrap_text=True, vertical="top"); ws.row_dimensions[3+i].height = min(240, 30 + 14 * (len(r["prompt"]) // 100))
ws.column_dimensions["E"].width = 100; ws.column_dimensions["B"].width = 22; ws.column_dimensions["G"].width = 30; ws.freeze_panes = "F4"
dv = DataValidation(type="list", formula1='"Y,N"', allow_blank=True); ws.add_data_validation(dv); dv.add(f"F4:F{3+len(rows_out)}")

ws2 = wb.create_sheet("Invented Shadow check")
ws2["A1"] = "Invented Shadow AI prompts the judge did NOT label Shadow AI. Mark use = N to drop one (a reserve of the same kind replaces it). Blank or Y keeps it."
ws2["A1"].font = Font(bold=True)
hdr2 = ["id", "role", "kind", "judge label", "judge conf", "prompt", "use (Y/N)", "notes"]
for c, h in enumerate(hdr2, 1): ws2.cell(row=3, column=c, value=h).font = Font(bold=True); ws2.cell(row=3, column=c).fill = PatternFill("solid", fgColor="FCE4D6")
off = [p for p in part if sl.get(p["id"], {}).get("label") != "Shadow AI"]
for i, p in enumerate(off, 1):
    t = p["type_text"] + ("\n\n" + p["paste_text"] if p["paste_text"] else "")
    for c, v in enumerate([p["id"], p["role"], p["kind"], sl.get(p["id"], {}).get("label"), sl.get(p["id"], {}).get("confidence"), t, "", ""], 1): ws2.cell(row=3+i, column=c, value=v)
    ws2.cell(row=3+i, column=6).alignment = Alignment(wrap_text=True, vertical="top")
ws2.column_dimensions["F"].width = 100
dv2 = DataValidation(type="list", formula1='"Y,N"', allow_blank=True); ws2.add_data_validation(dv2); dv2.add(f"G4:G{3+max(1, len(off))}")
wb.save(REVIEW_XLSX)
json.dump({"items": key, "quota": eff, "min_judge_conf": MIN_JUDGE_CONF, "prompt_version": PROMPT_VERSION, "mask_version": MASK_VERSION, "dataset": DATASET},
          open(REVIEW_XLSX.replace(".xlsx", "_key.json"), "w"), indent=1)
print(f"review workbook -> {REVIEW_XLSX}: {len(rows_out)} real rows ({dict(Counter(g for g, _, _ in rows_out))}), {len(off)} invented prompts to check")
print("pre-marked Y by length:", dict(Counter((g, "short" if short(r) else "long") for g, r, u in rows_out if u == "Y")))


In [ ]:
#@title 8) After review: assemble deploy/prompt_bank_v2.json, then rewrite the invented keys as documentation-style examples (does not touch prompt_bank.json)
# Add --force to the first line to rebuild after changing the review workbook.
!python "{ROOT}/code/build_prompt_bank_v2b.py" --p2 "{ROOT}"
# GitHub push protection blocks realistic fake keys; this rewrites only the key strings of invented prompts (AKIA...EXAMPLE etc.)
!python "{ROOT}/code/pool_v2_defang.py" "{ROOT}/deploy/prompt_bank_v2.json" "{ROOT}/deploy/prompt_bank_v2.json"
